# 📑 FolioPP: Fundamental & Corporate Filings Analyzer (NSE)

This notebook analyzes **Institutional Fundamental Data** directly from NSE India, including:
- 📢 **Announcements** & Corporate Filings
- 📅 **Board Meetings** & Upcoming Events
- 💰 **Financial Results** (Quarterly/Annual)
- 🤝 **Corporate Actions** (Dividends, Splits, Bonus)
- 📊 **Shareholding Patterns**

In [12]:
import os
import sys
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# === PATH CONFIGURATION ===
notebook_dir = os.getcwd()
backend_root = os.path.abspath(os.path.join(notebook_dir, '..'))

# Add NSE provider and Core and Root to sys.path
sys.path.insert(0, os.path.join(backend_root, 'providers', 'nse'))
sys.path.insert(0, os.path.join(backend_root, 'core'))
sys.path.insert(0, backend_root)

SYMBOL = "RELIANCE"   # NSE Symbol to analyze
DAYS = 90             # Lookback for filings/results

print(f"📌 Analysis Target: {SYMBOL}")
print(f"📂 Backend Path: {backend_root}")

📌 Analysis Target: RELIANCE
📂 Backend Path: d:\hackathons\ET\backend


In [13]:
# === CORE NSE FETCHER ===
import requests
from foliopp_nse.utils.helpers import nse_fetch, nse_json, to_nse_date, NSE_BASE

def fetch_filings(symbol, filing_type="announcements"):
    to_date = datetime.now(); from_date = to_date - timedelta(days=DAYS)
    endpoints = {
        "announcements": f"{NSE_BASE}/api/corporate-announcements",
        "board_meetings": f"{NSE_BASE}/api/event-calendar",
        "corp_actions": f"{NSE_BASE}/api/corporate-actions",
        "financial_results": f"{NSE_BASE}/api/corporate-financial-results",
        "shareholding": f"{NSE_BASE}/api/corporate-shareholding-pattern"
    }
    url = endpoints.get(filing_type)
    params = {"index": "equities", "symbol": symbol}
    if filing_type != "shareholding":
        params.update({"from_date": to_nse_date(from_date), "to_date": to_nse_date(to_date)})
    
    print(f"🔍 Fetching {filing_type} for {symbol}...")
    resp = nse_fetch(url, params=params)
    if resp.status_code == 200: return nse_json(resp)
    return []

## 1. 📢 Corporate Announcements
Displays recent filings related to company alerts, regulatory disclosures, and news.

In [14]:
data = fetch_filings(SYMBOL, "announcements")
if data:
    df = pd.DataFrame(data)
    # ROBUST SCHEMA HANDLING: Only select columns that actually exist
    possible_cols = {
        'an_dt': 'Date',
        'desc': 'Description',
        'subject': 'Category',
        'attachementFile': 'Attachment_Link',
        'att': 'Attachment_Link', # Alias sometimes used by NSE
        'dt': 'Date',            # Alternate date key
        'sm_name': 'Symbol'      # Symbol alias
    }
    cols_to_use = [c for c in possible_cols.keys() if c in df.columns]
    display_df = df[cols_to_use].rename(columns=possible_cols)
    display(display_df.head(10))
else:
    print("⚠️ No announcements found in last 90 days. Raw Response:", data)

🔍 Fetching announcements for RELIANCE...


,Date,Description,Date,Symbol
0,18-Mar-2026 00:49:38,Action(s) taken or orders passed,18032026004938,Reliance Industries Limited
1,16-Mar-2026 19:15:56,Updates,16032026191556,Reliance Industries Limited
2,10-Mar-2026 21:49:05,Updates,10032026214905,Reliance Industries Limited
3,10-Mar-2026 19:15:17,Updates,10032026191517,Reliance Industries Limited
4,09-Mar-2026 16:46:42,Updates,09032026164642,Reliance Industries Limited
5,06-Mar-2026 13:20:33,Updates,06032026132033,Reliance Industries Limited
6,04-Mar-2026 19:32:53,Analysts/Institutional Investor Meet/Con. Call...,04032026193253,Reliance Industries Limited
7,25-Feb-2026 19:50:54,Other Restructuring,25022026195054,Reliance Industries Limited
8,24-Feb-2026 19:49:23,Updates,24022026194923,Reliance Industries Limited
9,23-Feb-2026 19:46:45,Updates,23022026194645,Reliance Industries Limited


## 2. 💰 Financial Results & Board Meetings
Shows upcoming board meetings and historical quarterly performance.

In [15]:
results = fetch_filings(SYMBOL, "financial_results")
if results:
    df_res = pd.DataFrame(results)
    cols = ['from_date', 'to_date', 'income', 'expenditure', 'profitAfterTax', 'eps']
    display(df_res[[c for c in cols if c in df_res.columns]].head(8))
else:
    print("No financial results.")

meetings = fetch_filings(SYMBOL, "board_meetings")
if meetings:
    df_meet = pd.DataFrame(meetings)
    cols = ['bm_date', 'purpose', 'details']
    display(df_meet[[c for c in cols if c in df_meet.columns]].head(5))
else:
    print("No board meetings.")

🔍 Fetching financial_results for RELIANCE...
No financial results.
🔍 Fetching board_meetings for RELIANCE...


,purpose
0,Financial Results


## 3. 📊 Shareholding Pattern
Analyzes the ownership structure of the company (Promoters vs Public).

In [16]:
shp = fetch_filings(SYMBOL, "shareholding")
if shp:
    try:
        rows = []
        for category in ['promoter', 'public']:
            cat_data = shp.get(category, {})
            rows.append({"Category": category.capitalize(), "Percentage": cat_data.get("percent", 0)})
        df_shp = pd.DataFrame(rows)
        display(df_shp)
        import matplotlib.pyplot as plt
        plt.figure(figsize=(4,4)); plt.pie(df_shp['Percentage'], labels=df_shp['Category'], autopct='%1.1f%%', colors=['#2ecc71', '#3498db'])
        plt.title(f"Ownership Distribution: {SYMBOL}"); plt.show()
    except Exception as e:
        print(f"⚠️ SHP Data parse issue: {e}. Raw: {shp}")
else:
    print("⚠️ No Shareholding data available.")

🔍 Fetching shareholding for RELIANCE...
⚠️ No Shareholding data available.
